In [ ]:
import geopandas as gpd
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

In [ ]:
# path = "/data/uscuni-restricted/04_spatial_census/_modifed_merged_census_2021.parquet"
path = "/data/uscuni-restricted/04_spatial_census/_merged_census_2021.parquet"
path_total = "/data/uscuni-restricted/04_spatial_census/total.csv"

In [ ]:
def process_file(path, path_total):
    cols_no_division = [
        "Hustota obyvatel na obytnou plochu",
        "Počet obyvatel na dům",
        "Počet obyvatel na byt",
    ]

    # Load total population data
    total = pd.read_csv(path_total, dtype={"nadzsjd": str}, index_col=0)

    # Load main data
    data = gpd.read_parquet(path)

    # Merge data
    data_total = data.join(total)

    # Drop unnecessary columns
    data_relative = data_total.drop(data.columns[:12], axis=1)

    # Convert columns (except 'geometry') to float
    cols_numeric = data_relative.columns.drop("geometry")
    data_relative[cols_numeric] = data_relative[cols_numeric].astype(float)

    # Normalize by total population (except certain columns)
    population = data_relative["Obyvatelstvo celkem"].replace(
        0, np.nan
    )  # avoid division by zero
    cols_to_normalize = [
        col
        for col in cols_numeric
        if col not in cols_no_division + ["Obyvatelstvo celkem"]
    ]
    data_relative[cols_to_normalize] = data_relative[cols_to_normalize].div(
        population, axis=0
    )

    # Drop rows with NaNs after division
    data_relative = data_relative.dropna(subset=cols_to_normalize)

    # Clip extremely large or small values
    data_relative[cols_to_normalize] = data_relative[cols_to_normalize].clip(-1e6, 1e6)
    data_relative[cols_no_division] = data_relative[cols_no_division].clip(-1e6, 1e6)

    # Replace any remaining infinities with NaN and drop them
    data_relative.replace([np.inf, -np.inf], np.nan, inplace=True)
    data_relative.dropna(subset=cols_to_normalize + cols_no_division, inplace=True)

    # Scale columns
    scaler = StandardScaler()
    data_relative[cols_to_normalize] = scaler.fit_transform(
        data_relative[cols_to_normalize]
    )
    data_relative[cols_no_division] = scaler.fit_transform(
        data_relative[cols_no_division]
    )

    return data_relative

In [ ]:
data_relative = process_file(path, path_total)

In [ ]:
from shapely import box

v = "v10"
selected_level = 3

# Load grid as a GeoDataFrame (keep all columns and geometries intact)
grid = gpd.read_parquet("/data/uscuni-restricted/geometries/nadzsj_d.parquet").to_crs(
    epsg=3035
)

german_boundary = box(*grid.total_bounds)
regions = gpd.read_parquet("/data/uscuni-ulce/regions/cadastre_regions_hull.parquet")
german_regions = regions[regions.intersects(german_boundary)].index.values

cluster_mapping = pd.read_parquet(
    f"/data/uscuni-ulce/processed_data/clusters/cluster_mapping_{v}.pq"
)
level_7_map = cluster_mapping[selected_level].to_dict()

all_buildings = []

for region_id in german_regions:
    buildings = gpd.read_parquet(
        f"/data/uscuni-ulce/processed_data/clusters/clusters_{region_id}_{v}.pq",
        columns=["geometry", "final_without_noise"],
    )

    buildings["class_level_7"] = buildings["final_without_noise"].map(level_7_map)
    all_buildings.append(buildings)

all_buildings = pd.concat(all_buildings, ignore_index=True)

# Query spatial index using grid.geometry
inp, res = grid.sindex.query(all_buildings.centroid, predicate="intersects")

# Deduplicate buildings (inp) so each building is assigned to only one cell
duplicated_buildings = pd.Series(inp).duplicated()
inp = inp[~duplicated_buildings]
res = res[~duplicated_buildings]

assigned = all_buildings.iloc[inp].copy()
assigned["grid_pos"] = res

# 1. Count Level 7 class occurrences per grid cell
counts = (
    assigned.groupby(["grid_pos", "class_level_7"]).size().reset_index(name="count")
)

# 2. Extract most common class per cell
most_common = (
    counts.sort_values(["grid_pos", "count"], ascending=[True, False])
    .drop_duplicates(subset=["grid_pos"], keep="first")
    .copy()
)

# 3. Calculate metrics
grid_totals = assigned.groupby("grid_pos").size()
most_common["total_buildings"] = most_common["grid_pos"].map(grid_totals)
most_common["proportion"] = most_common["count"] / most_common["total_buildings"]
most_common["percentage"] = most_common["proportion"] * 100

# Map integer position back to the grid's original DataFrame index
most_common.index = grid.index[most_common["grid_pos"].values]

stats = most_common.rename(columns={"class_level_7": "most_common_level_7"})[
    ["most_common_level_7", "count", "total_buildings", "proportion", "percentage"]
]

# Keep all grid geometries (cells with no buildings will have NaN for stats)
grid_assignment = grid.join(stats)

# Alternatively, if you only want grid cells that contain buildings:
# grid_assignment = grid.join(stats, how="inner")

In [ ]:
grid_assignment

In [ ]:
grid_assignment = grid_assignment[grid_assignment.index.isin(data_relative.index)]
grid_assignment

In [ ]:
grid_assignment.describe()

In [ ]:
grid_assignment.loc[grid_assignment["percentage"] < 60]

In [ ]:
clusters = pd.read_csv(
    "/data/uscuni-restricted/04_spatial_census/cluster_assignment_v10.csv",
    dtype={"kod_nadzsj_d": str},
)
cluster_mapping = pd.read_parquet(
    "/data/uscuni-ulce/processed_data/clusters/cluster_mapping_v10.pq"
)
data = data_relative.merge(clusters, left_on="nadzsjd", right_on="kod_nadzsj_d")


data["Cluster"] = data["final_without_noise"].map(cluster_mapping[3])

In [ ]:
data = data.set_index("kod_nadzsj_d")

In [ ]:
data["Cluster_new"] = grid_assignment["most_common_level_7"]

In [ ]:
data["Cluster_new"] = data["Cluster_new"].astype(int)

In [ ]:
data["Cluster"]

In [ ]:
data["Cluster_new"]

In [ ]:
(data["Cluster"] == data["Cluster_new"]).value_counts()

In [ ]:
# data.loc[data["same"] == False].plot()

In [ ]:
changes = (
    data[data["Cluster"] != data["Cluster_new"]]
    .groupby(["Cluster", "Cluster_new"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)
print(changes)

In [ ]:
data["Obyvatelstvo celkem"].describe()

In [ ]:
deciles = data["Obyvatelstvo celkem"].quantile(np.linspace(0, 1, 11))
deciles